## LangGraph Open Deep Research - Supervisor-Researcher Architecture

In this notebook, we'll explore the **supervisor-researcher delegation architecture** for conducting deep research with LangGraph.

You can visit this repository to see the original application: [Open Deep Research](https://github.com/langchain-ai/open_deep_research)

Let's jump in!

## What We're Building

This implementation uses a **hierarchical delegation pattern** where:

1. **User Clarification** - Optionally asks clarifying questions to understand the research scope
2. **Research Brief Generation** - Transforms user messages into a structured research brief
3. **Supervisor** - A lead researcher that analyzes the brief and delegates research tasks
4. **Parallel Researchers** - Multiple sub-agents that conduct focused research simultaneously
5. **Research Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings are combined into a comprehensive report

![Architecture Diagram](https://private-user-images.githubusercontent.com/181020547/465824799-12a2371b-8be2-4219-9b48-90503eb43c69.png?jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmF3LmdpdGh1YnVzZXJjb250ZW50LmNvbSIsImtleSI6ImtleTUiLCJleHAiOjE3NjAwNDgyMzcsIm5iZiI6MTc2MDA0NzkzNywicGF0aCI6Ii8xODEwMjA1NDcvNDY1ODI0Nzk5LTEyYTIzNzFiLThiZTItNDIxOS05YjQ4LTkwNTAzZWI0M2M2OS5wbmc_WC1BbXotQWxnb3JpdGhtPUFXUzQtSE1BQy1TSEEyNTYmWC1BbXotQ3JlZGVudGlhbD1BS0lBVkNPRFlMU0E1M1BRSzRaQSUyRjIwMjUxMDA5JTJGdXMtZWFzdC0xJTJGczMlMkZhd3M0X3JlcXVlc3QmWC1BbXotRGF0ZT0yMDI1MTAwOVQyMjEyMTdaJlgtQW16LUV4cGlyZXM9MzAwJlgtQW16LVNpZ25hdHVyZT1iYTRmYTAzYjkzYjA2MGE4ZTZlYjQ4ODU1OWIwY2VlZWU0Mzk0YzdmMjQ1YTlhMDMyNmI3NWNlZTQxNDdlZGViJlgtQW16LVNpZ25lZEhlYWRlcnM9aG9zdCJ9.a8477QD1J4Lrmys7jB8gt_H5pdiKBsKsu3npEqZjEpo)

This differs from a section-based approach by allowing dynamic task decomposition based on the research question, rather than predefined sections.

![Alt text](data/image_1.png)
![Alt text](data/image_2.png)


## Dependencies

You'll need API keys for Anthropic (for the LLM) and Tavily (for web search). We'll configure the system to use Anthropic's Claude Sonnet 4 exclusively.

In [21]:
import os
import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

## Task 1: State Definitions

The state structure is hierarchical with three levels:

### Agent State (Top Level)
Contains the overall conversation messages, research brief, accumulated notes, and final report.

### Supervisor State (Middle Level)
Manages the research supervisor's messages, research iterations, and coordinating parallel researchers.

### Researcher State (Bottom Level)
Each individual researcher has their own message history, tool call iterations, and research findings.

We also have structured outputs for tool calling:
- **ConductResearch** - Tool for supervisor to delegate research to a sub-agent
- **ResearchComplete** - Tool to signal research phase is done
- **ClarifyWithUser** - Structured output for asking clarifying questions
- **ResearchQuestion** - Structured output for the research brief

Let's import these from our library: [`open_deep_library/state.py`](open_deep_library/state.py)

In [2]:
# Import state definitions from the library
from open_deep_library.state import (
    # Main workflow states
    AgentState,           # Lines 65-72: Top-level agent state with messages, research_brief, notes, final_report
    AgentInputState,      # Lines 62-63: Input state is just messages
    
    # Supervisor states
    SupervisorState,      # Lines 74-81: Supervisor manages research delegation and iterations
    
    # Researcher states
    ResearcherState,      # Lines 83-90: Individual researcher with messages and tool iterations
    ResearcherOutputState, # Lines 92-96: Output from researcher (compressed research + raw notes)
    
    # Structured outputs for tool calling
    ConductResearch,      # Lines 15-19: Tool for delegating research to sub-agents
    ResearchComplete,     # Lines 21-22: Tool to signal research completion
    ClarifyWithUser,      # Lines 30-41: Structured output for user clarification
    ResearchQuestion,     # Lines 43-48: Structured output for research brief
)

#### ❓ Question 1:

 Explain the interrelationships between the three states.  Why don't we just make a single huge state?

 Interrelationships
AgentState contains the overall conversation and final report
SupervisorState manages research delegation and coordination between researchers
ResearcherState handles individual focused research on specific topics
The states flow downward (Agent → Supervisor → Researchers) and upward (Researchers → Supervisor → Agent) as research progresses.
Why Not a Single Huge State?
Using separate states provides several critical advantages:

1. Separation of Concerns
Each state has a specific responsibility and clear boundaries
AgentState: Overall workflow and final output
SupervisorState: Research planning and delegation
ResearcherState: Individual research execution

2. Parallel Execution
Multiple ResearcherStates can run simultaneously without conflicts
Each researcher maintains independent state for their specific topic
No shared mutable state that could cause race conditions

3. Scalability
Token limits: Smaller states stay within model context windows
Memory efficiency: Only load relevant state for each subgraph
Error isolation: Failures in one researcher don't affect others

4. Maintainability
Easier debugging: Clear state boundaries make issues easier to trace
Modular design: Can modify one level without affecting others
Testing: Can test each state independently

5. Flexibility
Different models: Each level can use different models optimized for their task
Different tools: Researchers can have different tool sets
Dynamic scaling: Can spawn as many researchers as needed

## Task 2: Utility Functions and Tools

The system uses several key utilities:

### Search Tools
- **tavily_search** - Async web search with automatic summarization to stay within token limits
- Supports Anthropic native web search and Tavily API

### Reflection Tools
- **think_tool** - Allows researchers to reflect on their progress and plan next steps (ReAct pattern)

### Helper Utilities
- **get_all_tools** - Assembles the complete toolkit (search + MCP + reflection)
- **get_today_str** - Provides current date context for research
- Token limit handling utilities for graceful degradation

These are defined in [`open_deep_library/utils.py`](open_deep_library/utils.py)

In [3]:
# Import utility functions and tools from the library
from open_deep_library.utils import (
    # Search tool - Lines 43-136: Tavily search with automatic summarization
    tavily_search,
    
    # Reflection tool - Lines 219-244: Strategic thinking tool for ReAct pattern
    think_tool,
    
    # Tool assembly - Lines 569-597: Get all configured tools
    get_all_tools,
    
    # Date utility - Lines 872-879: Get formatted current date
    get_today_str,
    
    # Supporting utilities for error handling
    get_api_key_for_model,          # Lines 892-914: Get API keys from config or env
    is_token_limit_exceeded,         # Lines 665-701: Detect token limit errors
    get_model_token_limit,           # Lines 831-846: Look up model's token limit
    remove_up_to_last_ai_message,    # Lines 848-866: Truncate messages for retry
    anthropic_websearch_called,      # Lines 607-637: Detect Anthropic native search usage
    openai_websearch_called,         # Lines 639-658: Detect OpenAI native search usage
    get_notes_from_tool_calls,       # Lines 599-601: Extract notes from tool messages
)

### ❓ Question 2:  

What are the advantages and disadvantages of importing these components instead of including them in the notebook?

Advantages of Importing Components

1. Code Organization & Maintainability
Separation of concerns: Business logic separated from presentation/notebook
Reusability: Components can be used across multiple notebooks or applications
Version control: Easier to track changes in dedicated files vs notebook cells
Code review: Easier to review complex logic in dedicated files

2. Development Experience
IDE support: Full syntax highlighting, autocomplete, and debugging in .py files
Testing: Can write unit tests for individual components
Linting: Better code quality enforcement with tools like ruff
Refactoring: Easier to rename, move, or restructure code

3. Performance & Efficiency
Faster execution: No need to re-execute all cells when only one component changes
Memory efficiency: Components loaded once, not recreated on each cell execution
Caching: Can implement proper caching strategies in dedicated modules

4. Collaboration
Team development: Multiple developers can work on different components simultaneously
Code sharing: Components can be shared across projects or published as packages
Documentation: Easier to generate API documentation from .py files

5. Production Readiness
Deployment: Components can be packaged and deployed independently
Configuration: Easier to manage different configurations for different environments
Monitoring: Better logging and monitoring capabilities in dedicated modules

Disadvantages of Importing Components

1. Learning Curve
Abstraction: Students need to understand the imported code without seeing it directly
Debugging difficulty: Harder to step through imported code during learning
Context switching: Need to jump between notebook and source files

2. Development Complexity
Dependency management: Need to ensure all imports are available
Version conflicts: Potential issues with different versions of imported components
Circular imports: Risk of import cycles in complex architectures

3. Transparency Issues
Black box effect: Students can't see the implementation details easily
Hidden complexity: Important implementation details might be overlooked
Learning barriers: Makes it harder to understand how things work under the hood

4. Notebook-Specific Challenges
Cell execution order: Imported components might not reflect latest changes
State management: Harder to inspect and modify state in imported components
Interactive debugging: Less convenient than having code directly in cells

Best Practices for This Educational Context

Hybrid Approach
Import core components (state, configuration, prompts) for production readiness
Show key implementations in notebook cells for educational purposes
Provide clear documentation explaining what each imported component does
Educational Enhancements
Include code references in comments (e.g., # Lines 60-115 in deep_researcher.py)
Provide inline explanations of what imported functions do
Show simplified versions of complex logic in notebook cells
Development Workflow
Start with notebook cells for rapid prototyping and learning
Refactor to modules once concepts are understood
Use imports for production-ready implementations

Conclusion:
For this educational project, importing components is beneficial because it:
Teaches production-ready patterns students will use in real projects
Demonstrates proper code organization and separation of concerns
Shows how to build modular, maintainable AI systems
However, the notebook compensates for the disadvantages by:
Providing detailed explanations of each imported component
Including code references to specific lines in source files
Offering hands-on activities to experiment with the system

## Task 3: Configuration System

The configuration system controls:

### Research Behavior
- **allow_clarification** - Whether to ask clarifying questions before research
- **max_concurrent_research_units** - How many parallel researchers can run (default: 5)
- **max_researcher_iterations** - How many times supervisor can delegate research (default: 6)
- **max_react_tool_calls** - Tool call limit per researcher (default: 10)

### Model Configuration
- **research_model** - Model for research and supervision (we'll use Anthropic)
- **compression_model** - Model for synthesizing findings
- **final_report_model** - Model for writing the final report
- **summarization_model** - Model for summarizing web search results

### Search Configuration
- **search_api** - Which search API to use (ANTHROPIC, TAVILY, or NONE)
- **max_content_length** - Character limit before summarization

Defined in [`open_deep_library/configuration.py`](open_deep_library/configuration.py)

In [4]:
# Import configuration from the library
from open_deep_library.configuration import (
    Configuration,    # Lines 38-247: Main configuration class with all settings
    SearchAPI,        # Lines 11-17: Enum for search API options (ANTHROPIC, TAVILY, NONE)
)

## Task 4: Prompt Templates

The system uses carefully engineered prompts for each phase:

### Phase 1: Clarification
**clarify_with_user_instructions** - Analyzes if the research scope is clear or needs clarification

### Phase 2: Research Brief
**transform_messages_into_research_topic_prompt** - Converts user messages into a detailed research brief

### Phase 3: Supervisor
**lead_researcher_prompt** - System prompt for the supervisor that manages delegation strategy

### Phase 4: Researcher
**research_system_prompt** - System prompt for individual researchers conducting focused research

### Phase 5: Compression
**compress_research_system_prompt** - Prompt for synthesizing research findings without losing information

### Phase 6: Final Report
**final_report_generation_prompt** - Comprehensive prompt for writing the final report

All prompts are defined in [`open_deep_library/prompts.py`](open_deep_library/prompts.py)

In [5]:
# Import prompt templates from the library
from open_deep_library.prompts import (
    clarify_with_user_instructions,                    # Lines 3-41: Ask clarifying questions
    transform_messages_into_research_topic_prompt,     # Lines 44-77: Generate research brief
    lead_researcher_prompt,                            # Lines 79-136: Supervisor system prompt
    research_system_prompt,                            # Lines 138-183: Researcher system prompt
    compress_research_system_prompt,                   # Lines 186-222: Research compression prompt
    final_report_generation_prompt,                    # Lines 228-308: Final report generation
)

## Task 5: Node Functions - The Building Blocks

Now let's look at the node functions that make up our graph. We'll import them from the library and understand what each does.

### The Complete Research Workflow

The workflow consists of 8 key nodes organized into 3 subgraphs:

1. **Main Graph Nodes:**
   - `clarify_with_user` - Entry point that checks if clarification is needed
   - `write_research_brief` - Transforms user input into structured research brief
   - `final_report_generation` - Synthesizes all research into final report

2. **Supervisor Subgraph Nodes:**
   - `supervisor` - Lead researcher that plans and delegates
   - `supervisor_tools` - Executes supervisor's tool calls (delegation, reflection)

3. **Researcher Subgraph Nodes:**
   - `researcher` - Individual researcher conducting focused research
   - `researcher_tools` - Executes researcher's tool calls (search, reflection)
   - `compress_research` - Synthesizes researcher's findings

All nodes are defined in [`open_deep_library/deep_researcher.py`](open_deep_library/deep_researcher.py)

### Node 1: clarify_with_user

**Purpose:** Analyzes user messages and asks clarifying questions if the research scope is unclear.

**Key Steps:**
1. Check if clarification is enabled in configuration
2. Use structured output to analyze if clarification is needed
3. If needed, end with a clarifying question for the user
4. If not needed, proceed to research brief with verification message

**Implementation:** [`open_deep_library/deep_researcher.py` lines 60-115](open_deep_library/deep_researcher.py#L60-L115)

In [6]:
# Import the clarify_with_user node
from open_deep_library.deep_researcher import clarify_with_user

### Node 2: write_research_brief

**Purpose:** Transforms user messages into a structured research brief for the supervisor.

**Key Steps:**
1. Use structured output to generate detailed research brief from messages
2. Initialize supervisor with system prompt and research brief
3. Set up supervisor messages with proper context

**Why this matters:** A well-structured research brief helps the supervisor make better delegation decisions.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 118-175](open_deep_library/deep_researcher.py#L118-L175)

In [7]:
# Import the write_research_brief node
from open_deep_library.deep_researcher import write_research_brief

### Node 3: supervisor

**Purpose:** Lead research supervisor that plans research strategy and delegates to sub-researchers.

**Key Steps:**
1. Configure model with three tools:
   - `ConductResearch` - Delegate research to a sub-agent
   - `ResearchComplete` - Signal that research is done
   - `think_tool` - Strategic reflection before decisions
2. Generate response based on current context
3. Increment research iteration count
4. Proceed to tool execution

**Decision Making:** The supervisor uses `think_tool` to reflect before delegating research, ensuring thoughtful decomposition of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 178-223](open_deep_library/deep_researcher.py#L178-L223)

In [8]:
# Import the supervisor node (from supervisor subgraph)
from open_deep_library.deep_researcher import supervisor

### Node 4: supervisor_tools

**Purpose:** Executes the supervisor's tool calls, including strategic thinking and research delegation.

**Key Steps:**
1. Check exit conditions:
   - Exceeded maximum iterations
   - No tool calls made
   - `ResearchComplete` called
2. Process `think_tool` calls for strategic reflection
3. Execute `ConductResearch` calls in parallel:
   - Spawn researcher subgraphs for each delegation
   - Limit to `max_concurrent_research_units` (default: 5)
   - Gather all results asynchronously
4. Aggregate findings and return to supervisor

**Parallel Execution:** This is where the magic happens - multiple researchers work simultaneously on different aspects of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 225-349](open_deep_library/deep_researcher.py#L225-L349)

In [9]:
# Import the supervisor_tools node
from open_deep_library.deep_researcher import supervisor_tools

### Node 5: researcher

**Purpose:** Individual researcher that conducts focused research on a specific topic.

**Key Steps:**
1. Load all available tools (search, MCP, reflection)
2. Configure model with tools and researcher system prompt
3. Generate response with tool calls
4. Increment tool call iteration count

**ReAct Pattern:** Researchers use `think_tool` to reflect after each search, deciding whether to continue or provide their answer.

**Available Tools:**
- Search tools (Tavily or Anthropic native search)
- `think_tool` for strategic reflection
- `ResearchComplete` to signal completion
- MCP tools (if configured)

**Implementation:** [`open_deep_library/deep_researcher.py` lines 365-424](open_deep_library/deep_researcher.py#L365-L424)

In [10]:
# Import the researcher node (from researcher subgraph)
from open_deep_library.deep_researcher import researcher

### Node 6: researcher_tools

**Purpose:** Executes the researcher's tool calls, including searches and strategic reflection.

**Key Steps:**
1. Check early exit conditions (no tool calls, native search used)
2. Execute all tool calls in parallel:
   - Search tools fetch and summarize web content
   - `think_tool` records strategic reflections
   - MCP tools execute external integrations
3. Check late exit conditions:
   - Exceeded `max_react_tool_calls` (default: 10)
   - `ResearchComplete` called
4. Continue research loop or proceed to compression

**Error Handling:** Safely handles tool execution errors and continues with available results.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 435-509](open_deep_library/deep_researcher.py#L435-L509)

In [11]:
# Import the researcher_tools node
from open_deep_library.deep_researcher import researcher_tools

### Node 7: compress_research

**Purpose:** Compresses and synthesizes research findings into a concise, structured summary.

**Key Steps:**
1. Configure compression model
2. Add compression instruction to messages
3. Attempt compression with retry logic:
   - If token limit exceeded, remove older messages
   - Retry up to 3 times
4. Extract raw notes from tool and AI messages
5. Return compressed research and raw notes

**Why Compression?** Researchers may accumulate lots of tool outputs and reflections. Compression ensures:
- All important information is preserved
- Redundant information is deduplicated
- Content stays within token limits for the final report

**Token Limit Handling:** Gracefully handles token limit errors by progressively truncating messages.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 511-585](open_deep_library/deep_researcher.py#L511-L585)

In [12]:
# Import the compress_research node
from open_deep_library.deep_researcher import compress_research

### Node 8: final_report_generation

**Purpose:** Generates the final comprehensive research report from all collected findings.

**Key Steps:**
1. Extract all notes from completed research
2. Configure final report model
3. Attempt report generation with retry logic:
   - If token limit exceeded, truncate findings by 10%
   - Retry up to 3 times
4. Return final report or error message

**Token Limit Strategy:**
- First retry: Use model's token limit × 4 as character limit
- Subsequent retries: Reduce by 10% each time
- Graceful degradation with helpful error messages

**Report Quality:** The prompt guides the model to create well-structured reports with:
- Proper headings and sections
- Inline citations
- Comprehensive coverage of all findings
- Sources section at the end

**Implementation:** [`open_deep_library/deep_researcher.py` lines 607-697](open_deep_library/deep_researcher.py#L607-L697)

In [13]:
# Import the final_report_generation node
from open_deep_library.deep_researcher import final_report_generation

## Task 6: Graph Construction - Putting It All Together

The system is organized into three interconnected graphs:

### 1. Researcher Subgraph (Bottom Level)
Handles individual focused research on a specific topic:
```
START → researcher → researcher_tools → compress_research → END
               ↑            ↓
               └────────────┘ (loops until max iterations or ResearchComplete)
```

### 2. Supervisor Subgraph (Middle Level)
Manages research delegation and coordination:
```
START → supervisor → supervisor_tools → END
            ↑              ↓
            └──────────────┘ (loops until max iterations or ResearchComplete)
            
supervisor_tools spawns multiple researcher_subgraphs in parallel
```

### 3. Main Deep Researcher Graph (Top Level)
Orchestrates the complete research workflow:
```
START → clarify_with_user → write_research_brief → research_supervisor → final_report_generation → END
                 ↓                                       (supervisor_subgraph)
               (may end early if clarification needed)
```

Let's import the compiled graphs from the library.

In [14]:
# Import the pre-compiled graphs from the library
from open_deep_library.deep_researcher import (
    # Bottom level: Individual researcher workflow
    researcher_subgraph,    # Lines 588-605: researcher → researcher_tools → compress_research
    
    # Middle level: Supervisor coordination
    supervisor_subgraph,    # Lines 351-363: supervisor → supervisor_tools (spawns researchers)
    
    # Top level: Complete research workflow
    deep_researcher,        # Lines 699-719: Main graph with all phases
)

## Why This Architecture?

### Advantages of Supervisor-Researcher Delegation

1. **Dynamic Task Decomposition**
   - Unlike section-based approaches with predefined structure, the supervisor can break down research based on the actual question
   - Adapts to different types of research (comparisons, lists, deep dives, etc.)

2. **Parallel Execution**
   - Multiple researchers work simultaneously on different aspects
   - Much faster than sequential section processing
   - Configurable parallelism (1-20 concurrent researchers)

3. **ReAct Pattern for Quality**
   - Researchers use `think_tool` to reflect after each search
   - Prevents excessive searching and improves search quality
   - Natural stopping conditions based on information sufficiency

4. **Flexible Tool Integration**
   - Easy to add MCP tools for specialized research
   - Supports multiple search APIs (Anthropic, Tavily)
   - Each researcher can use different tool combinations

5. **Graceful Token Limit Handling**
   - Compression prevents token overflow
   - Progressive truncation in final report generation
   - Research can scale to arbitrary depths

### Trade-offs

- **Complexity:** More moving parts than section-based approach
- **Cost:** Parallel researchers use more tokens (but faster)
- **Unpredictability:** Research structure emerges dynamically

## Task 7: Running the Deep Researcher

Now let's see the system in action! We'll use it to analyze a PDF document about how people use AI.

### Setup

We need to:
1. Load the PDF document
2. Configure the execution with Anthropic settings
3. Run the research workflow

In [15]:
# Load the PDF document
from pathlib import Path
import PyPDF2

def load_pdf(pdf_path: str) -> str:
    """Load and extract text from PDF."""
    pdf_text = ""
    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page in pdf_reader.pages:
            pdf_text += page.extract_text() + "\n\n"
    return pdf_text

# Load the PDF about how people use AI
pdf_path = "data/howpeopleuseai.pdf"
pdf_content = load_pdf(pdf_path)

print(f"Loaded PDF with {len(pdf_content)} characters")
print(f"First 500 characters:\n{pdf_content[:500]}...")

Loaded PDF with 112460 characters
First 500 characters:
NBER WORKING PAPER SERIES
HOW PEOPLE USE CHATGPT
Aaron Chatterji
Thomas Cunningham
David J. Deming
Zoe Hitzig
Christopher Ong
Carl Yan Shan
Kevin Wadman
Working Paper 34255
http://www.nber.org/papers/w34255
NATIONAL BUREAU OF ECONOMIC RESEARCH
1050 Massachusetts Avenue
Cambridge, MA 02138
September 2025
We acknowledge help and comments from Joshua Achiam, Hemanth Asirvatham, Ryan 
Beiermeister,  Rachel Brown, Cassandra Duchan Solis, Jason Kwon, Elliott Mokski, Kevin Rao, 
Harrison Satcher,  Gawe...


In [16]:
# Set up the graph with Anthropic configuration
from IPython.display import Markdown, display
import uuid

# Note: deep_researcher is already compiled from the library
# For this demo, we'll use it directly without additional checkpointing
graph = deep_researcher

print("✓ Graph ready for execution")
print("  (Note: The graph is pre-compiled from the library)")

✓ Graph ready for execution
  (Note: The graph is pre-compiled from the library)


### Configuration for Anthropic

We'll configure the system to use:
- **Claude Sonnet 4** for all research, supervision, and report generation
- **Tavily** for web search (you can also use Anthropic's native search)
- **Moderate parallelism** (3 concurrent researchers)
- **Clarification enabled** (will ask if research scope is unclear)

In [17]:
# Configure for Anthropic with moderate settings
config = {
    "configurable": {
        # Model configuration - using Claude Sonnet 4 for everything
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        
        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,
        
        # Research behavior
        "allow_clarification": True,
        "max_concurrent_research_units": 1,  # 1 parallel researchers
        "max_researcher_iterations": 2,      # Supervisor can delegate up to 2 times
        "max_react_tool_calls": 3,           # Each researcher can make up to 3 tool calls
        
        # Search configuration
        "search_api": "tavily",  # Using Tavily for web search
        "max_content_length": 50000,
        
        # Thread ID for this conversation
        "thread_id": str(uuid.uuid4())
    }
}

print("✓ Configuration ready")
print(f"  - Research Model: Claude Sonnet 4")
print(f"  - Max Concurrent Researchers: 3")
print(f"  - Max Iterations: 4")
print(f"  - Search API: Tavily")

✓ Configuration ready
  - Research Model: Claude Sonnet 4
  - Max Concurrent Researchers: 3
  - Max Iterations: 4
  - Search API: Tavily


### Execute the Research

Now let's run the research! We'll ask the system to analyze the PDF and provide insights about how people use AI.

The workflow will:
1. **Clarify** - Check if the request is clear (may skip if obvious)
2. **Research Brief** - Transform our request into a structured brief
3. **Supervisor** - Plan research strategy and delegate to researchers
4. **Parallel Research** - Multiple researchers gather information simultaneously
5. **Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings combined into comprehensive report

In [27]:
# Create our research request with PDF context
research_request = f"""
I have a PDF document about how people use AI. Please analyze this document and provide insights about:

1. What are the main findings about how people are using AI?
2. What are the most common use cases?
3. What trends or patterns emerge from the data?

Here's the PDF content:

{pdf_content[:10000]}  # First 10k chars to stay within limits

...[content truncated for context window]
"""

# Execute the graph
async def run_research():
    """Run the research workflow and display results."""
    print("Starting research workflow...\n")
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        config,
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)

# Run the research
await run_research()

Starting research workflow...


Node: clarify_with_user

I have sufficient information to analyze the provided PDF document about ChatGPT usage patterns. I understand you want insights on: (1) main findings about how people use AI, (2) most common use cases, and (3) emerging trends and patterns from the data. The document appears to be an NBER working paper titled "How People Use ChatGPT" with comprehensive data on usage patterns, demographics, and conversation classifications. I will now begin analyzing this research to provide you with detailed insights on these three key areas.

Node: write_research_brief

Research Brief Generated:
I have an NBER working paper titled "How People Use ChatGPT" by Chatterji et al. (2025) that analyzes ChatGPT usage patterns from November 2022 through July 2025. I need a comprehensive analysis of this document focusing on three specific areas: (1) What are the main findings about how people are using AI/ChatGPT - including adoption rates, demographic pa

# Comprehensive Analysis of ChatGPT Usage Patterns: Research Limitations and Available Insights

Based on the research brief requesting analysis of the NBER working paper "How People Use ChatGPT" by Chatterji et al. (2025), this report addresses the technical limitations encountered during the research process and provides analysis based on the document excerpt provided in the initial messages.

## Research Limitations and Technical Constraints

The comprehensive analysis requested could not be completed due to technical limitations that prevented access to external search capabilities. Specifically, API configuration issues blocked the ability to retrieve the full NBER working paper or additional sources that would have provided complete data on ChatGPT usage patterns from November 2022 through July 2025.

However, the initial message history contains substantial content from the paper's introduction and abstract, which provides valuable insights into the three requested areas of analysis.

## Main Findings About AI/ChatGPT Usage

### Unprecedented Adoption Scale and Speed
ChatGPT achieved remarkable adoption rates, reaching approximately 700 million users by July 2025, representing around 10% of the global adult population. The platform processes 18 billion messages weekly, demonstrating what the researchers describe as having "no precedent" in terms of speed of global diffusion for a new technology.

### Dramatic Shift from Work to Non-Work Usage
A significant finding highlighted in the research is the substantial shift in usage patterns over time. The data shows that non-work-related messages grew from 53% in June 2024 to over 70% by June 2025, representing a fundamental change in how people utilize the platform. This shift occurred within existing user cohorts rather than being driven by changes in the composition of new users.

The absolute numbers demonstrate massive growth across both categories:
- Non-work messages: 238 million daily in June 2024 to 1,911 million daily in June 2025
- Work messages: 213 million daily in June 2024 to 716 million daily in June 2025

### Demographic Evolution Patterns
The research reveals important demographic trends in ChatGPT adoption:
- **Gender Gap Narrowing**: Early adopters were disproportionately male, but the gender gap has narrowed dramatically over the study period
- **Geographic Distribution**: Higher growth rates observed in lower-income countries, suggesting global accessibility and adoption
- **Educational and Occupational Patterns**: Work usage correlates more strongly with educated users in highly-paid professional occupations

## Most Common Use Cases and Conversation Categories

### The Dominant Three: 80% of All Usage
The research identifies three primary categories that collectively account for nearly 80% of all ChatGPT conversations:

**1. Practical Guidance**
This emerges as the most common use case, encompassing:
- Tutoring and teaching activities
- How-to advice across various topics
- Creative ideation and brainstorming

**2. Seeking Information**
This category functions as a close substitute for traditional web search, including:
- Information searches about people and current events
- Product research and comparisons
- Recipe searches and recommendations

**3. Writing**
This category dominates work-related tasks and includes:
- Automated production of emails and documents
- Text editing, critiquing, and summarizing
- Translation services
- Content creation for various communications

### Work vs. Non-Work Use Case Distribution
Writing represents the most prevalent work-related use case, accounting for 40% of work-related messages in June 2025. This highlights ChatGPT's unique ability to generate digital outputs, distinguishing it from traditional search engines.

### Less Common but Notable Use Cases
The research indicates that computer programming and self-expression represent relatively small shares of overall usage, contrary to some public perceptions about AI chatbot applications.

## Trends and Patterns from the Data

### Economic Value Proposition
The research suggests that ChatGPT provides significant economic value primarily through decision support, which proves especially important in knowledge-intensive jobs. This finding aligns with estimated consumer surplus of at least $97 billion in 2024 alone in the United States.

### Usage Evolution Within Cohorts
A particularly interesting finding is that the shift toward non-work usage occurred within existing user cohorts rather than being driven by new user acquisition. This suggests that as users become more familiar with the platform, they expand their usage beyond initial work-focused applications.

### Implications for Economic Analysis
The research notes that while most economic analysis of AI focuses on productivity impacts in paid work, the impact on non-work activities (home production) appears to be on a similar scale and possibly larger. This finding suggests that traditional economic impact assessments may underestimate the full value of AI chatbots.

### Global Diffusion Patterns
The higher growth rates in lower-income countries indicate that ChatGPT is achieving significant global penetration beyond traditional early-adopter demographics, potentially democratizing access to AI-powered assistance tools.

## Research Methodology and Privacy Considerations

The study employed sophisticated methodological approaches to analyze usage patterns while protecting user privacy:
- Automated classification pipelines using privacy-preserving protocols
- Secure data clean room protocols for demographic analysis
- Representative sampling from consumer plans (Free, Plus, Pro)
- Multiple classification taxonomies including work/non-work, conversation topics, and interaction types

## Limitations of This Analysis

This analysis is constrained by the inability to access the complete research paper due to technical limitations. The findings presented are based solely on the abstract and introduction sections provided in the initial messages. A complete analysis would require access to:
- Detailed demographic breakdowns and statistical analyses
- Comprehensive trend data across the full study period
- Specific quantitative findings about user group differences
- Complete methodology and validation procedures
- Discussion and conclusion sections with broader implications

### Sources

Due to technical limitations preventing successful search execution, no external sources were accessible for this analysis. The insights provided are derived from the document excerpt included in the initial message history, specifically from the NBER Working Paper No. 34255 "How People Use ChatGPT" by Chatterji et al., published in September 2025.


Research workflow completed!


## Understanding the Output

Let's break down what happened:

### Phase 1: Clarification
The system checked if your request was clear. Since you provided a PDF and specific questions, it likely proceeded without clarification.

### Phase 2: Research Brief
Your request was transformed into a detailed research brief that guides the supervisor's delegation strategy.

### Phase 3: Supervisor Delegation
The supervisor analyzed the brief and decided how to break down the research:
- Used `think_tool` to plan strategy
- Called `ConductResearch` multiple times to delegate to parallel researchers
- Each delegation specified a focused research topic

### Phase 4: Parallel Research
Multiple researchers worked simultaneously:
- Each researcher used web search tools to gather information
- Used `think_tool` to reflect after each search
- Decided when they had enough information
- Compressed their findings into clean summaries

### Phase 5: Final Report
All research findings were synthesized into a comprehensive report with:
- Well-structured sections
- Inline citations
- Sources listed at the end
- Balanced coverage of all findings

#### 🏗️ Activity #1: Try Different Configurations

You can experiment with different settings to see how they affect the research.  You may select three or more of the following settings (or invent your own experiments) and describe the results.

### Increase Parallelism
```python
"max_concurrent_research_units": 10  # More researchers working simultaneously
```

### Deeper Research
```python
"max_researcher_iterations": 8   # Supervisor can delegate more times
"max_react_tool_calls": 15      # Each researcher can search more
```

### Use Anthropic Native Search
```python
"search_api": "anthropic"  # Use Claude's built-in web search
```

### Disable Clarification
```python
"allow_clarification": False  # Skip clarification phase
```

## Key Takeaways

### Architecture Benefits
1. **Dynamic Decomposition** - Research structure emerges from the question, not predefined
2. **Parallel Efficiency** - Multiple researchers work simultaneously
3. **ReAct Quality** - Strategic reflection improves search decisions
4. **Scalability** - Handles token limits gracefully through compression
5. **Flexibility** - Easy to add new tools and capabilities

### When to Use This Pattern
- **Complex research questions** that need multi-angle investigation
- **Comparison tasks** where parallel research on different topics is beneficial
- **Open-ended exploration** where structure should emerge dynamically
- **Time-sensitive research** where parallel execution speeds up results

### When to Use Section-Based Instead
- **Highly structured reports** with predefined format requirements
- **Template-based content** where sections are always the same
- **Sequential dependencies** where later sections depend on earlier ones
- **Budget constraints** where token efficiency is critical

Test Configuration 1 - Increase Parallelism


In [19]:
# Activity #1: Test Different Configurations
# We'll test 3 different configurations and compare results

import time
from IPython.display import Markdown, display

# Create a simple research request for testing
test_request = """
Analyze the current trends in AI research and development. Focus on:
1. Key breakthroughs in 2024-2025
2. Major companies and their AI strategies
3. Emerging applications and use cases
"""

print("🧪 Starting Configuration Testing Activity")
print("="*60)

🧪 Starting Configuration Testing Activity


In [22]:
# Configuration 1: High Parallelism
print("\n🔬 Configuration 1: High Parallelism")
print("-" * 40)

config_high_parallel = {
    "configurable": {
        # Model configuration
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,
        
        # HIGH PARALLELISM SETTINGS
        "allow_clarification": False,  # Skip clarification for faster testing
        "max_concurrent_research_units": 5,  # 5 parallel researchers
        "max_researcher_iterations": 3,      # Supervisor can delegate up to 3 times
        "max_react_tool_calls": 5,           # Each researcher can make up to 5 tool calls
        
        # Search configuration
        "search_api": "tavily",
        "max_content_length": 50000,
        "thread_id": str(uuid.uuid4())
    }
}

print("Settings:")
print(f"  - Max Concurrent Researchers: 5")
print(f"  - Max Iterations: 3")
print(f"  - Max Tool Calls per Researcher: 5")
print(f"  - Clarification: Disabled")

# Run the test
start_time = time.time()
async for event in graph.astream(
    {"messages": [{"role": "user", "content": test_request}]},
    config_high_parallel,
    stream_mode="updates"
):
    for node_name, node_output in event.items():
        if node_name == "supervisor_tools" and "notes" in node_output:
            print(f"  📝 Research notes collected: {len(node_output['notes'])}")
        elif node_name == "final_report_generation" and "final_report" in node_output:
            print(f"  ✅ Final report generated")
            high_parallel_report = node_output["final_report"]

end_time = time.time()
high_parallel_time = end_time - start_time
print(f"  ⏱️  Total time: {high_parallel_time:.2f} seconds")


🔬 Configuration 1: High Parallelism
----------------------------------------
Settings:
  - Max Concurrent Researchers: 5
  - Max Iterations: 3
  - Max Tool Calls per Researcher: 5
  - Clarification: Disabled
  ✅ Final report generated
  ⏱️  Total time: 146.57 seconds


Test Configuration 2 - Deeper Research

In [23]:
# Configuration 2: Deeper Research
print("\n🔬 Configuration 2: Deeper Research")
print("-" * 40)

config_deeper_research = {
    "configurable": {
        # Model configuration
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,
        
        # DEEPER RESEARCH SETTINGS
        "allow_clarification": False,
        "max_concurrent_research_units": 2,  # Fewer parallel researchers
        "max_researcher_iterations": 6,      # Supervisor can delegate up to 6 times
        "max_react_tool_calls": 10,         # Each researcher can make up to 10 tool calls
        
        # Search configuration
        "search_api": "tavily",
        "max_content_length": 50000,
        "thread_id": str(uuid.uuid4())
    }
}

print("Settings:")
print(f"  - Max Concurrent Researchers: 2")
print(f"  - Max Iterations: 6")
print(f"  - Max Tool Calls per Researcher: 10")
print(f"  - Clarification: Disabled")

# Run the test
start_time = time.time()
async for event in graph.astream(
    {"messages": [{"role": "user", "content": test_request}]},
    config_deeper_research,
    stream_mode="updates"
):
    for node_name, node_output in event.items():
        if node_name == "supervisor_tools" and "notes" in node_output:
            print(f"  📝 Research notes collected: {len(node_output['notes'])}")
        elif node_name == "final_report_generation" and "final_report" in node_output:
            print(f"  ✅ Final report generated")
            deeper_research_report = node_output["final_report"]

end_time = time.time()
deeper_research_time = end_time - start_time
print(f"  ⏱️  Total time: {deeper_research_time:.2f} seconds")


🔬 Configuration 2: Deeper Research
----------------------------------------
Settings:
  - Max Concurrent Researchers: 2
  - Max Iterations: 6
  - Max Tool Calls per Researcher: 10
  - Clarification: Disabled
  ✅ Final report generated
  ⏱️  Total time: 154.52 seconds


Test Configuration 3 - Anthropic Native Search


In [24]:
# Configuration 3: Anthropic Native Search
print("\n🔬 Configuration 3: Anthropic Native Search")
print("-" * 40)

config_anthropic_search = {
    "configurable": {
        # Model configuration
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,
        
        # ANTHROPIC NATIVE SEARCH SETTINGS
        "allow_clarification": False,
        "max_concurrent_research_units": 3,  # Moderate parallelism
        "max_researcher_iterations": 4,      # Supervisor can delegate up to 4 times
        "max_react_tool_calls": 6,         # Each researcher can make up to 6 tool calls
        
        # Search configuration - USING ANTHROPIC NATIVE SEARCH
        "search_api": "anthropic",  # Changed from "tavily" to "anthropic"
        "max_content_length": 50000,
        "thread_id": str(uuid.uuid4())
    }
}

print("Settings:")
print(f"  - Max Concurrent Researchers: 3")
print(f"  - Max Iterations: 4")
print(f"  - Max Tool Calls per Researcher: 6")
print(f"  - Search API: Anthropic Native")
print(f"  - Clarification: Disabled")

# Run the test
start_time = time.time()
async for event in graph.astream(
    {"messages": [{"role": "user", "content": test_request}]},
    config_anthropic_search,
    stream_mode="updates"
):
    for node_name, node_output in event.items():
        if node_name == "supervisor_tools" and "notes" in node_output:
            print(f"  📝 Research notes collected: {len(node_output['notes'])}")
        elif node_name == "final_report_generation" and "final_report" in node_output:
            print(f"  ✅ Final report generated")
            anthropic_search_report = node_output["final_report"]

end_time = time.time()
anthropic_search_time = end_time - start_time
print(f"  ⏱️  Total time: {anthropic_search_time:.2f} seconds")


🔬 Configuration 3: Anthropic Native Search
----------------------------------------
Settings:
  - Max Concurrent Researchers: 3
  - Max Iterations: 4
  - Max Tool Calls per Researcher: 6
  - Search API: Anthropic Native
  - Clarification: Disabled
  ✅ Final report generated
  ⏱️  Total time: 118.66 seconds


 Compare Results

In [25]:
# Compare Results
print("\n📊 Configuration Comparison Results")
print("="*60)

print(f"Configuration 1 (High Parallelism):")
print(f"  ⏱️  Time: {high_parallel_time:.2f} seconds")
print(f"  📄 Report length: {len(high_parallel_report)} characters")
print(f"  🔍 Search method: Tavily API")

print(f"\nConfiguration 2 (Deeper Research):")
print(f"  ⏱️  Time: {deeper_research_time:.2f} seconds")
print(f"  📄 Report length: {len(deeper_research_report)} characters")
print(f"  🔍 Search method: Tavily API")

print(f"\nConfiguration 3 (Anthropic Native Search):")
print(f"  ⏱️  Time: {anthropic_search_time:.2f} seconds")
print(f"  📄 Report length: {len(anthropic_search_report)} characters")
print(f"  🔍 Search method: Anthropic Native")

# Analysis
print(f"\n🔍 Analysis:")
fastest_config = min([
    ("High Parallelism", high_parallel_time),
    ("Deeper Research", deeper_research_time),
    ("Anthropic Native", anthropic_search_time)
], key=lambda x: x[1])

longest_report = max([
    ("High Parallelism", len(high_parallel_report)),
    ("Deeper Research", len(deeper_research_report)),
    ("Anthropic Native", len(anthropic_search_report))
], key=lambda x: x[1])

print(f"  🏃 Fastest: {fastest_config[0]} ({fastest_config[1]:.2f}s)")
print(f"  📚 Most comprehensive: {longest_report[0]} ({longest_report[1]} chars)")


📊 Configuration Comparison Results
Configuration 1 (High Parallelism):
  ⏱️  Time: 146.57 seconds
  📄 Report length: 10987 characters
  🔍 Search method: Tavily API

Configuration 2 (Deeper Research):
  ⏱️  Time: 154.52 seconds
  📄 Report length: 4062 characters
  🔍 Search method: Tavily API

Configuration 3 (Anthropic Native Search):
  ⏱️  Time: 118.66 seconds
  📄 Report length: 627 characters
  🔍 Search method: Anthropic Native

🔍 Analysis:
  🏃 Fastest: Anthropic Native (118.66s)
  📚 Most comprehensive: High Parallelism (10987 chars)


Display Sample Reports

In [26]:
# Display sample reports for comparison
print("\n📋 Sample Reports Comparison")
print("="*60)

print("\n🔬 High Parallelism Report (First 500 chars):")
print("-" * 40)
print(high_parallel_report[:500] + "...")

print("\n🔬 Deeper Research Report (First 500 chars):")
print("-" * 40)
print(deeper_research_report[:500] + "...")

print("\n🔬 Anthropic Native Search Report (First 500 chars):")
print("-" * 40)
print(anthropic_search_report[:500] + "...")


📋 Sample Reports Comparison

🔬 High Parallelism Report (First 500 chars):
----------------------------------------
# Comprehensive Analysis of Current Trends in AI Research and Development

## Research Limitations and Context

This comprehensive analysis of AI research and development trends from 2024-2025 encountered significant technical limitations during the research process. All attempted searches for current primary sources—including recent company announcements, research papers, industry reports, and official documentation—were unsuccessful due to API configuration issues. As a result, this analysis i...

🔬 Deeper Research Report (First 500 chars):
----------------------------------------
# Comprehensive Analysis of AI Research and Development Trends (2024-2025)

## Research Limitations and Technical Constraints

Unfortunately, due to technical limitations with the search functionality during this research process, I was unable to access the current primary sources and real-tim

## Next Steps

### Extend the System
1. **Add MCP Tools** - Integrate specialized tools for your domain
2. **Custom Prompts** - Modify prompts for specific research types
3. **Different Models** - Try different Claude versions or mix models
4. **Persistence** - Use a real database for checkpointing instead of memory

### Learn More
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [Open Deep Research Repo](https://github.com/langchain-ai/open_deep_research)
- [Anthropic Claude Documentation](https://docs.anthropic.com/)
- [Tavily Search API](https://tavily.com/)

### Deploy
- Use LangGraph Cloud for production deployment
- Add proper error handling and logging
- Implement rate limiting and cost controls
- Monitor research quality and costs